In [1]:
import sys
sys.path.append("../")

import cv2 as cv
import numpy as np
from pathlib import Path

from sulllam.pipeline import SLAMPipeline, SLAMConfig
from sulllam.localization.extraction.orb import ORBFeatureExtractor, ORBConfigs
from sulllam.localization.matching.bf import BFFeatureMatcher, BFMatcherConfig
from sulllam.localization.pose_estimation.eight_point_estimator import EightPointPoseEstimator, EightPointEstimatorConfig
from sulllam.mapping.bundle_adjustment.local_bundle_adjustment import LocalBundleAdjustment, LocalBundleAdjustmentConfig
from sulllam.utils.ros import ROSPublisherWrapper

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Images:

In [2]:
image_dir = Path("/home/zhukowych/Projects/ucu/MMML/SULLLAM/data/itspace")
images = [cv.imread(image_path) for image_path in sorted(image_dir.glob("*.png"))[::15]]
len(images)

79

Camera calibration:

In [3]:
K = np.array([[533.340727445877, 0.0, 254.64689387916482],
              [0.0, 533.2556495307942, 256.4835490935692],
              [0.0, 0.0, 1.0]])

xi = np.array([[1.73241756065]])

D = np.array([-0.05972430882700243, 0.17468739202093328, 0.000737218969875311, 0.000574074894976456])

fx, fy = 2740.0, 2740.0
cx, cy = 2016.0, 1512.0

# Define the K matrix
K = np.array([
    [1638.35,    0.0,  960.0],
    [   0.0, 1575.81,  540.0],
    [   0.0,    0.0,    1.0]
])

## Configure & Run SLAM Pipeline

In [4]:
ros_publisher = ROSPublisherWrapper()

config = SLAMConfig(
    K=K,

    # Swap any component by passing an instance here, e.g.:
    # extractor=ORBFeatureExtractor(ORBConfigs(nfeatures=2000)),
    # matcher=BFFeatureMatcher(BFMatcherConfig()),
    # pose_estimator=EightPointPoseEstimator(EightPointEstimatorConfig(K=K)),
    bundle_adjustment=LocalBundleAdjustment(LocalBundleAdjustmentConfig(window_size=10, max_iterations=20, huber_radius=1.0)),

    max_reproj_error=2.0,
    max_depth=50.0,
    max_points=200,
    ba_frequency=5,
    ba_min_frames=15,
    clouds_dir=Path("clouds"),
)

In [5]:
pipeline = SLAMPipeline(config)
trajectory = pipeline.run(images, ros_publisher=ros_publisher)


[BA] --- Starting Local Bundle Adjustment ---
[BA] Total keyframes: 15
[BA] Local keyframe IDs (with observations): [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[BA] 1052 points, 3828 window obs, 603 external obs from 5 fixed external KFs
[BA] Anchored (fixed) window keyframes: [5, 6]
[BA] Graph: 589 fixed-cam edges (window), 3239 opt-cam edges, 603 external fixed-cam edges
[BA] Initial error: 34836.2549
[BA] Status      : NonlinearOptimizerStatus.CONVERGED
[BA] Iterations  : 14
[BA] Final error : 4831.5229  (Δ 30004.7319)
[BA] --- Local Bundle Adjustment Complete ---


[BA] --- Starting Local Bundle Adjustment ---
[BA] Total keyframes: 20
[BA] Local keyframe IDs (with observations): [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[BA] 1115 points, 4198 window obs, 875 external obs from 7 fixed external KFs
[BA] Anchored (fixed) window keyframes: [10, 11]
[BA] Graph: 898 fixed-cam edges (window), 3300 opt-cam edges, 875 external fixed-cam edges
[BA] Initial error: 18968.8049
[BA] Status      : Nonlin

In [6]:
ros_publisher.shutdown()

[ROS] Node shut down.
